# The Physics Problem: Parasitic ZZ Crosstalk
Hardware-oriented Quantum Machine Learning (hardware-aware QML) is a major focus of active research. For superconducting architectures (like the ones hosted on Amazon Braket from Rigetti or Oxford Quantum Circuits), gates are abstract representations of real-world analog microwave pulses.

One of the largest roadblocks in scaling these QPUs is parasitic crosstalk (specifically, unwanted always-on $ZZ$ interactions). Because qubits are physically wired next to each other on a chip's coupling map, driving a pulse on Qubit 0 inadvertently shifts the frequency and phase of Qubit 1.

Instead of writing digital gates, we can use Differentiable Pulse Programming via the qml.pulse module. We will design a time-dependent drive Hamiltonian, simulate hardware noise, and use classical machine learning to optimize the shape and amplitude of the microwave pulse to naturally suppress parasitic crosstalk across a coupling map edge.

## The Physics Problem: Parasitic ZZ Crosstalk
When two transmon qubits are coupled, their combined physical Hamiltonian contains a passive error term:$$H_{\text{coupling}} = J (Z_0 \otimes Z_1)$$If we try to perform a simple single-qubit $X$ rotation on Qubit 0 by applying a microwave drive, this background $J$ coupling continuously corrupts the state of Qubit 1. We will train a neural network/optimizer to find a pulse profile that performs the target rotation on Qubit 0 while forcing the net execution error on Qubit 1 to zero.


> [Control Line] ───>  Qubit 0  ◄─── Parasitic Coupling (J) ───►  Qubit 1 (Victim)

Note: The qml.pulse module utilizes jax as its backend engine. Ensure you have jax and jaxlib installed (pip install jax jaxlib) to run this simulation pipeline.



In [ ]:
# Install required packages (quiet mode for cleaner output)
!pip install --quiet pennylane
!pip install --quiet amazon-braket-pennylane-plugin

### Step 1: Defining the Noisy Hardware Environment

In [ ]:
import pennylane as qml
import jax
import jax.numpy as jnp

# Enable float64 for precise differential ordinary differential equation (ODE) solving
jax.config.update("jax_enable_x64", True)

num_wires = 2
duration = 1.5  # Duration of the microwave pulse injection in nanoseconds

# 1. Define the Hardware Drift Hamiltonian (Always-on Parasitic Crosstalk)
# Let's assume a strong parasitic coupling of J = 0.4 GHz between Qubit 0 and 1
J_coupling = 0.4
H_drift = J_coupling * (qml.PauliZ(0) @ qml.PauliZ(1))

# 2. Define the Drive Hamiltonian on Qubit 0
# We use a parameterized piecewise-constant function to model the shape of the pulse amplitude
def pulse_amplitude(params, t):
    # Divide the total duration into 4 distinct time bins
    bin_idx = jnp.minimum(jnp.int32(len(params) * t / duration), len(params) - 1)
    return params[bin_idx]

H_drive = pulse_amplitude * qml.PauliX(0)

# Combine into a Parametrized Hamiltonian
H_total = H_drift + H_drive

##Step 2: The Pulse Simulation QNode

In [ ]:
dev = qml.device("default.qubit", wires=num_wires)

@qml.qnode(dev, interface="jax")
def simulate_pulse_hardware(params):
    # Start the system in the ground state |00>
    # Evolve the system under the time-dependent pulse from t=0 to t=duration
    qml.evolve(H_total)(params, t=[0, duration])

    # Return the full state vector to evaluate crosstalk leakage
    return qml.state()

###Step 3: Differentiating the Hardware Error (The Loss Function)
Our target goal is to perform a perfect $\pi$-pulse (NOT gate) on Qubit 0 ($|0\rangle \rightarrow |1\rangle$), while keeping Qubit 1 safely unperturbed in its ground state ($|0\rangle$). Thus, our ideal target final state is exactly $|10\rangle$.

In [ ]:
# Target state vector corresponding to |10>
target_state = jnp.array([0.0, 0.0, 1.0, 0.0], dtype=jnp.complex128)

def hardware_loss_fn(params):
    final_state = simulate_pulse_hardware(params)

    # Infidelity Loss = 1 - |<Final_State | Target_State>|^2
    fidelity = jnp.abs(jnp.vdot(final_state, target_state)) ** 2
    return 1.0 - fidelity

###Step 4: Optimizing the Pulse Profile with Gradient Descent

In [ ]:
# Initialize 4 flat pulse amplitude segments (random voltages)
initial_pulse_params = jnp.array([1.0, 2.0, 1.5, 0.5], dtype=jnp.float64)

# Standard Gradient Descent tracking loop via JAX
params = initial_pulse_params
learning_rate = 0.2

print("Beginning Pulse Optimization to Mitigate Coupling Map Noise...")
print("-----------------------------------------------------------------")

for step in range(31):
    loss_val = hardware_loss_fn(params)

    # Use JAX to calculate the exact hardware gradient through the ODE solver
    grads = jax.grad(hardware_loss_fn)(params)

    # Update pulse amplitudes
    params -= learning_rate * grads

    if step % 10 == 0:
        print(f"Step {step:2d} | Infidelity Loss: {loss_val:.6f} | Pulse Amplitudes: {np.array(params)}")

print("-----------------------------------------------------------------")
print("Pulse Calibration Optimization Complete!")

###Step 5: Verification of Noise Suppression

In [ ]:
final_state = simulate_pulse_hardware(params)
state_probabilities = jnp.abs(final_state) ** 2

print("\n--- Hardware State Probability Readout ---")
print(f"Probability of staying in |00> (No operation): {state_probabilities[0]*100:.2f}%")
print(f"Probability of leakage to |01> (Crosstalk Error): {state_probabilities[1]*100:.2f}%")
print(f"Probability of hitting |10> (Target Perfect State): {state_probabilities[2]*100:.2f}%")
print(f"Probability of hitting |11> (Double Error):       {state_probabilities[3]*100:.2f}%")

## Breaking Down the Research Potential
This exact format is how physical characterization teams at companies like Rigetti calibrate their native gates. If your students want to adapt this blueprint into a research project, here is how they scale it:

**Scaling up the Coupling Map Complexity**
Instead of 2 qubits, extend this simulation to a 3-qubit linear chain ($Q_0 \leftrightarrow Q_1 \leftrightarrow Q_2$) or a heavy-hex layout. If you drive a pulse on $Q_1$, it leaks crosstalk in both directions simultaneously. Students can optimize a single pulse profile that creates a localized gate on $Q_1$ while creating a "decoherence shield" on both neighbors.

**DRAG Pulse Optimization (Derivative Removal by Adiabatic Gate)**
In actual transmon hardware, another prominent source of noise is leakage out of the computational basis entirely into higher-excited states (like the $|2\rangle$ state of the anharmonic oscillator). To combat this, researchers use DRAG pulses, where the main pulse is sent on the $X$ channel, and a secondary, derivative-shaped corrective pulse is sent simultaneously on the $Y$ channel:$$\Omega_Y(t) = -\frac{\alpha}{\Delta} \frac{d\Omega_X(t)}{dt}$$Students can modify H_total to include both a parameterized $X$ drive and a $Y$ drive, then task the optimizer with tracking down the ideal $\alpha$ coefficient that cancels out state leakage under a given coupling configuration.

**Open Quantum Systems (Lindblad Master Equation)**
The model above uses pure states. In a real physical cryostat, environmental thermal relaxation ($T_1$ decay) and dephasing ($T_2$ noise) continuously degrade performance. Instead of a standard state simulator, students can integrate PennyLane's pulse module with an open-quantum-system framework to optimize control pulses that can actively "outrun" the characteristic $T_1$ relaxation times of physical hardware devices.